This notebook covers post-unification CGM processing on the unified subject-level dataframe. We reshape CGM series across granularities (subject, daily, individual recordings) useful for downstream analyses and run quality checks (duplicates, missing values, coverage, temporal continuity).

In [1]:
import sys
import os

parent_dir = os.path.abspath("../")
sys.path.append(parent_dir)

import polars as pl
from chronoindex.dataset_loader import load_datasets
from chronoindex.glucose_series_processing.ts_manager import TimeSeriesManager
from chronoindex.dataset_unifier import UnifiedCGMDataset, UnifiedFoodData, UnifiedPhysicalActivityData

In [2]:
datasets, errors = load_datasets(
    config_path= "datasets.toml",
    only=["zhao","praes_unil"]
)
if errors:
    raise RuntimeError(errors)

➡️ Loading datasets.zhao
➡️ Loading datasets.praes_unil


In [3]:
unified = UnifiedCGMDataset(
    datasets
    )

df = unified.unified_data   # same as unified.cgm_data
df.head()

Id,CGM,CGMTime,dataset
str,list[f64],list[datetime[μs]],str
"""2000_0_20201230""","[138.6, 120.6, … 88.2]","[2020-12-30 13:54:00, 2020-12-30 14:09:00, … 2021-01-13 12:24:00]","""zhao"""
"""2001_0_20201102""","[102.6, 97.2, … 136.8]","[2020-11-02 09:40:00, 2020-11-02 09:55:00, … 2020-11-14 20:10:00]","""zhao"""
"""2001_1_20201117""","[151.2, 145.8, … 180.0]","[2020-11-17 09:19:00, 2020-11-17 09:34:00, … 2020-12-01 07:49:00]","""zhao"""
"""2002_0_20210513""","[257.4, 246.6, … 217.8]","[2021-05-13 16:22:00, 2021-05-13 16:37:00, … 2021-05-19 14:52:00]","""zhao"""
"""2003_0_20210615""","[264.6, 262.8, … 180.0]","[2021-06-15 17:20:00, 2021-06-15 17:35:00, … 2021-06-21 19:20:00]","""zhao"""


In [4]:
ts_manager = TimeSeriesManager(df)
ts_manager.to_individual_recordings(patient_identifier_col="Id")
recordings_df = ts_manager.df
recordings_df

Id,CGM,CGMTime,dataset
str,f64,datetime[μs],str
"""s241081147""",97.2972,2024-10-10 11:23:00,"""praes_unil"""
"""s241081147""",95.4954,2024-10-10 11:38:00,"""praes_unil"""
"""s241081147""",95.4954,2024-10-10 11:53:00,"""praes_unil"""
"""s241081147""",95.4954,2024-10-10 12:08:00,"""praes_unil"""
"""s241081147""",93.6936,2024-10-10 12:23:00,"""praes_unil"""
…,…,…,…
"""2099_0_20201116""",97.2,2020-11-30 07:47:00,"""zhao"""
"""2099_0_20201116""",93.6,2020-11-30 08:02:00,"""zhao"""
"""2099_0_20201116""",90.0,2020-11-30 08:17:00,"""zhao"""


In [5]:
ts_manager = TimeSeriesManager(df)
ts_manager.to_daily_df(patient_identifier_col="Id", temporal_resolution="hourly")
print(ts_manager.df)

shape: (74_261, 6)
┌────────────┬─────────────────┬──────┬────────────┬───────────────────────┬───────────────────────┐
│ dataset    ┆ Id              ┆ hour ┆ date       ┆ CGM                   ┆ CGMTime               │
│ ---        ┆ ---             ┆ ---  ┆ ---        ┆ ---                   ┆ ---                   │
│ str        ┆ str             ┆ i8   ┆ date       ┆ list[f64]             ┆ list[datetime[μs]]    │
╞════════════╪═════════════════╪══════╪════════════╪═══════════════════════╪═══════════════════════╡
│ praes_unil ┆ s24425831       ┆ 13   ┆ 2024-07-02 ┆ [97.2972, 100.9008, … ┆ [2024-07-02 13:04:00, │
│            ┆                 ┆      ┆            ┆ 104.5044…             ┆ 2024-07-…             │
│ praes_unil ┆ s24111587       ┆ 4    ┆ 2025-01-25 ┆ [81.081, 81.081, …    ┆ [2025-01-25 04:00:00, │
│            ┆                 ┆      ┆            ┆ 82.8828]              ┆ 2025-01-…             │
│ zhao       ┆ 2055_0_20210524 ┆ 10   ┆ 2021-05-24 ┆ [199.8, 190.8, 183.

In [6]:
ts_manager.to_subject_df(patient_identifier_col="Id")
print(ts_manager.df)

shape: (185, 4)
┌────────────┬─────────────────┬────────────────────────────────┬─────────────────────────────────┐
│ dataset    ┆ Id              ┆ CGM                            ┆ CGMTime                         │
│ ---        ┆ ---             ┆ ---                            ┆ ---                             │
│ str        ┆ str             ┆ list[f64]                      ┆ list[datetime[μs]]              │
╞════════════╪═════════════════╪════════════════════════════════╪═════════════════════════════════╡
│ zhao       ┆ 2034_0_20210624 ┆ [246.6, 250.2, … 97.2]         ┆ [2021-06-24 12:31:00, 2021-06-… │
│ praes_unil ┆ s24711712       ┆ [88.2882, 104.5044, … 84.6846] ┆ [2024-07-17 13:03:00, 2024-07-… │
│ praes_unil ┆ s25116857       ┆ [91.8918, 93.6936, … 95.4954]  ┆ [2025-02-11 10:18:00, 2025-02-… │
│ zhao       ┆ 2064_0_20210608 ┆ [145.8, 145.8, … 219.6]        ┆ [2021-06-08 14:55:00, 2021-06-… │
│ zhao       ┆ 2099_0_20201116 ┆ [86.4, 77.4, … 86.4]           ┆ [2020-11-16 12:47:

In [7]:
ts_manager = TimeSeriesManager(df)
ts_manager.to_daily_df(patient_identifier_col="Id", temporal_resolution="day_night")
print(ts_manager.df)

shape: (6_424, 6)
┌────────────┬─────────────────┬───────┬────────────┬───────────────────────┬──────────────────────┐
│ dataset    ┆ Id              ┆ AM_PM ┆ date       ┆ CGM                   ┆ CGMTime              │
│ ---        ┆ ---             ┆ ---   ┆ ---        ┆ ---                   ┆ ---                  │
│ str        ┆ str             ┆ bool  ┆ date       ┆ list[f64]             ┆ list[datetime[μs]]   │
╞════════════╪═════════════════╪═══════╪════════════╪═══════════════════════╪══════════════════════╡
│ praes_unil ┆ s2412161129     ┆ true  ┆ 2024-12-24 ┆ [102.7026, 95.4954, … ┆ [2024-12-24          │
│            ┆                 ┆       ┆            ┆ 97.2972]              ┆ 13:13:00, 2024-12-…  │
│ zhao       ┆ 2079_0_20210809 ┆ false ┆ 2021-08-10 ┆ [59.4, 75.6, … 127.8] ┆ [2021-08-10          │
│            ┆                 ┆       ┆            ┆                       ┆ 00:04:00, 2021-08-…  │
│ praes_unil ┆ s24111587       ┆ false ┆ 2024-11-30 ┆ [88.2882, 84.6846, 

In [8]:
ts_manager = TimeSeriesManager(df)
ts_manager.to_daily_df(patient_identifier_col="Id", temporal_resolution="daily")
print(ts_manager.df)

shape: (3_371, 5)
┌────────────┬─────────────────┬────────────┬─────────────────────────┬───────────────────────┐
│ dataset    ┆ Id              ┆ date       ┆ CGM                     ┆ CGMTime               │
│ ---        ┆ ---             ┆ ---        ┆ ---                     ┆ ---                   │
│ str        ┆ str             ┆ date       ┆ list[f64]               ┆ list[datetime[μs]]    │
╞════════════╪═════════════════╪════════════╪═════════════════════════╪═══════════════════════╡
│ praes_unil ┆ s24819736       ┆ 2024-11-11 ┆ [86.4864, 84.6846, …    ┆ [2024-11-11 00:13:00, │
│            ┆                 ┆            ┆ 115.3152]               ┆ 2024-11-…             │
│ praes_unil ┆ s24819759       ┆ 2024-11-15 ┆ [88.2882, 84.6846, …    ┆ [2024-11-15 00:13:00, │
│            ┆                 ┆            ┆ 102.7026]               ┆ 2024-11-…             │
│ praes_unil ┆ s24527852       ┆ 2024-05-31 ┆ [82.8828, 61.2612, …    ┆ [2024-05-31 00:03:00, │
│            ┆        

Taking CGMs around other available information

In [9]:
ubeh = UnifiedFoodData(datasets)
food_df = ubeh.unified_data

In [10]:
ts_manager = TimeSeriesManager(df)  
out = ts_manager.cgms_around_events(
    events_df=food_df,
    event_column="FoodTimepoint",
    patient_identifier_col="Id",
    hour_window=2.0,
)
out.head()

Id,food,FoodTimepoint,meal,calories,carbs,proteins,fats,dataset,CGMBefore,CGMTimeBefore,CGMAfter,CGMTimeAfter
str,str,datetime[μs],str,f64,f64,f64,f64,str,list[f64],list[datetime[μs]],list[f64],list[datetime[μs]]
"""s241081147""","""ŪDENS, KRĀNA ŪDENS (200 ml). O…",2024-10-10 08:00:00,"""Breakfast""",634.9095,22.411,19.3885,48.429,"""praes_unil""",null,null,null,null
"""s241081147""","""CŪKGAĻAS KARBONĀDE, KOTLETES (…",2024-10-10 12:30:00,"""Lunch""",766.652,5.48,79.64,45.336,"""praes_unil""","[97.2972, 95.4954, … 93.6936]","[2024-10-10 11:23:00, 2024-10-10 11:38:00, … 2024-10-10 12:23:00]","[88.2882, 86.4864, … 97.2972]","[2024-10-10 12:38:00, 2024-10-10 12:53:00, … 2024-10-10 14:23:00]"
"""s241081147""","""LAPU SALĀTI (100 g). LASIS, KĀ…",2024-10-10 15:30:00,"""Dinner""",1095.826,66.8985,42.2055,71.789,"""praes_unil""","[79.2792, 93.6936, … 86.4864]","[2024-10-10 13:38:00, 2024-10-10 13:53:00, … 2024-10-10 15:23:00]","[86.4864, 82.8828, … 81.081]","[2024-10-10 15:38:00, 2024-10-10 15:53:00, … 2024-10-10 17:23:00]"
"""s241081147""","""ŪDENS, KRĀNA ŪDENS (100 ml)""",2024-10-11 03:20:20.824,"""Breakfast""",0.0,0.0,0.0,0.0,"""praes_unil""",null,null,null,null
"""s241081147""","""TOMĀTI (250 g)""",2024-10-11 06:44:26.451,"""Breakfast""",56.825,8.625,1.375,0.75,"""praes_unil""",null,null,null,null


In [11]:
ubeh = UnifiedPhysicalActivityData(datasets)
act_df = ubeh.unified_data

In [12]:
act_df

Id,activity_name,startTime,duration,ActivityTimepoint,dataset
str,str,str,i64,datetime[μs],str
"""s241081147""","""Walk""","""16:20""",2099000,2024-10-10 16:20:00,"""praes_unil"""
"""s241081147""","""Walk""","""19:16""",1075000,2024-10-10 19:16:00,"""praes_unil"""
"""s241081147""","""Walk""","""20:08""",1331000,2024-10-10 20:08:00,"""praes_unil"""
"""s241081147""","""Walk""","""06:48""",1485000,2024-10-11 06:48:00,"""praes_unil"""
"""s241081147""","""Walk""","""16:50""",2713000,2024-10-11 16:50:00,"""praes_unil"""
…,…,…,…,…,…
"""s2512892""","""Run""","""19:31""",3549000,2025-06-05 19:31:00,"""praes_unil"""
"""s2512892""","""Walk""","""15:48""",6450000,2025-06-07 15:48:00,"""praes_unil"""
"""s2512892""","""Run""","""10:00""",3518000,2025-06-08 10:00:00,"""praes_unil"""


In [13]:
out = ts_manager.cgms_around_events(
    events_df=act_df,
    event_column="ActivityTimepoint",
    patient_identifier_col="Id",
    hour_window=2.0,
)
out.head()

Id,activity_name,startTime,duration,ActivityTimepoint,dataset,CGMBefore,CGMTimeBefore,CGMAfter,CGMTimeAfter
str,str,str,i64,datetime[μs],str,list[f64],list[datetime[μs]],list[f64],list[datetime[μs]]
"""s241081147""","""Walk""","""16:20""",2099000,2024-10-10 16:20:00,"""praes_unil""","[97.2972, 93.6936, … 75.6756]","[2024-10-10 14:23:00, 2024-10-10 14:38:00, … 2024-10-10 16:08:00]","[72.072, 73.8738, … 79.2792]","[2024-10-10 16:23:00, 2024-10-10 16:38:00, … 2024-10-10 18:08:00]"
"""s241081147""","""Walk""","""19:16""",1075000,2024-10-10 19:16:00,"""praes_unil""","[81.081, 81.081, … 73.8738]","[2024-10-10 17:23:00, 2024-10-10 17:38:00, … 2024-10-10 19:08:00]","[72.072, 77.4774, … 88.2882]","[2024-10-10 19:23:00, 2024-10-10 19:38:00, … 2024-10-10 21:08:00]"
"""s241081147""","""Walk""","""20:08""",1331000,2024-10-10 20:08:00,"""praes_unil""","[79.2792, 81.081, … 82.8828]","[2024-10-10 18:08:00, 2024-10-10 18:23:00, … 2024-10-10 20:08:00]","[86.4864, 88.2882, … 91.8918]","[2024-10-10 20:23:00, 2024-10-10 20:38:00, … 2024-10-10 22:08:00]"
"""s241081147""","""Walk""","""06:48""",1485000,2024-10-11 06:48:00,"""praes_unil""",null,null,null,null
"""s241081147""","""Walk""","""16:50""",2713000,2024-10-11 16:50:00,"""praes_unil""",null,null,null,null


Running quality checks on the cgm

In [14]:
from chronoindex.glucose_series_processing.cgm_checking_functions import run_all_unified_checks

results = run_all_unified_checks(
    df,
    min_unique_days=7,
    min_records=1000,
)

Running unified CGM checks
- Input shape: rows=185, cols=4
- Grouping key: 'dataset + Id'
- Thresholds: min_unique_days=7, min_records=1000
[1/5] Checking duplicate CGM readings...
No duplicate CGM readings found.
[2/5] Checking null glucose/timestamp values...
Null glucose/timestamp values found for the following subjects:
shape: (5, 5)
┌─────────┬─────────────────┬─────────────────────┬──────────────────┬────────────────────┐
│ dataset ┆ Id              ┆ null_glucose_values ┆ null_time_values ┆ rows_with_any_null │
│ ---     ┆ ---             ┆ ---                 ┆ ---              ┆ ---                │
│ str     ┆ str             ┆ u32                 ┆ u32              ┆ u32                │
╞═════════╪═════════════════╪═════════════════════╪══════════════════╪════════════════════╡
│ zhao    ┆ 2023_0_20210812 ┆ 3                   ┆ 0                ┆ 3                  │
│ zhao    ┆ 2040_0_20210729 ┆ 1                   ┆ 0                ┆ 1                  │
│ zhao    ┆ 2080